# Tutorial 4: Automatic serial and parallel generation

`GenerateData.generate()` selects the workflow automatically. With no `max_obs`, generation is serial and returns data in memory. When `max_obs` is smaller than the requested observation count, generation is parallel, writes chunks to disk, and returns `(None, None)`.

Both workflows are compared below after loading their results into memory. The comparison checks identical logical output: dimensions, variable names, shapes, and observation counts. Random values and coordinates are generated independently per chunk, so byte-for-byte equality is not expected.

In [ ]:
from pathlib import Path
import shutil

import dask.dataframe as dd
import pandas as pd
import xarray as xr

from data_sparsity.generate_data import GenerateData

output_dir = Path("tutorial4_output")
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir()


## 1. Single-variable generation

The first generator is serial because `max_obs` is not provided. The second is parallel because `max_obs=25` creates multiple chunks.

The parallel path generates coordinates and random values independently within each chunk. Therefore, serial and parallel numeric values and coordinate values are not expected to be byte-for-byte equal, even for a single variable. The comparison below verifies the guarantees that matter for the generated dataset: identical dimensions, variables, shapes, and non-NaN observation counts.

In [ ]:
single_config = dict(
    num_obs=100,
    num_dims=1,
    ratio_dims=1,
    density=1.0,
    seed=34,
)

serial_single = GenerateData(**single_config)
serial_single_ds, serial_single_df = serial_single.generate()
serial_single_ds = serial_single_ds.to_dataset(name="record")

single_nc = output_dir / "single" / "data.nc"
single_pq = output_dir / "single" / "data"
single_tmp = output_dir / "single" / "tmp" / "chunk.parquet"
parallel_single = GenerateData(**single_config, max_obs=25)
parallel_result = parallel_single.generate(
    netcdf_filepath=str(single_nc),
    parquet_filepath=str(single_pq),
    parquet_tmp=str(single_tmp),
)

assert serial_single.NTASKS == 1
assert parallel_single.NTASKS > 1
assert parallel_result == (None, None)

parallel_single_files = list(single_nc.parent.glob("data_*.nc"))
parallel_single_ds = xr.open_mfdataset(parallel_single_files)
parallel_single_df = dd.read_parquet(single_pq.parent).compute()
assert serial_single_ds.sizes == parallel_single_ds.sizes
assert set(serial_single_ds.data_vars) == set(parallel_single_ds.data_vars)
assert serial_single_ds["record"].shape == parallel_single_ds["record"].shape
assert int(serial_single_ds["record"].count()) == int(parallel_single_ds["record"].count())
assert set(serial_single_df.columns) == set(parallel_single_df.columns)
assert len(serial_single_df) == len(parallel_single_df)
assert int(serial_single_df["record"].notna().sum()) == int(parallel_single_df["record"].notna().sum())
print("Single-variable serial and parallel results have identical logical structure and occupancy.")

## 2. Multi-variable generation

The same automatic selection works for multiple variables. The serial result is returned directly, while the parallel result is loaded from standard NetCDF and Parquet readers.

In [ ]:
multi_config = dict(
    num_obs=80,
    num_dims=3,
    ratio_dims=1,
    density=0.15,
    seed=42,
    num_vars=2,
    var_dims=2,
    overlap=[0.5],
    fixed_overlap=True,
)

serial_multi = GenerateData(**multi_config)
serial_multi_ds, serial_multi_df = serial_multi.generate()

multi_nc = output_dir / "multi" / "data.nc"
multi_pq = output_dir / "multi" / "data"
multi_tmp = output_dir / "multi" / "tmp" / "chunk.parquet"
parallel_multi = GenerateData(**multi_config, max_obs=20)
parallel_result = parallel_multi.generate(
    netcdf_filepath=str(multi_nc),
    parquet_filepath=str(multi_pq),
    parquet_tmp=str(multi_tmp),
)

assert serial_multi.NTASKS == 1
assert parallel_multi.NTASKS > 1
assert parallel_result == (None, None)

parallel_multi_files = list(multi_nc.parent.glob("data_*.nc"))
parallel_multi_ds = xr.open_mfdataset(parallel_multi_files)
parallel_multi_df = dd.read_parquet(multi_pq.parent).compute()

assert serial_multi_ds.sizes == parallel_multi_ds.sizes
assert set(serial_multi_ds.data_vars) == set(parallel_multi_ds.data_vars)
for name in serial_multi_ds.data_vars:
    assert parallel_multi_ds[name].dims == tuple(f"x{i}" for i in range(parallel_multi.num_dims))
    assert parallel_multi_ds[name].shape == tuple(parallel_multi.shape)
    assert serial_multi_ds[name].count() == parallel_multi_ds[name].count()

assert set(serial_multi_df.columns) == set(parallel_multi_df.columns)
for name in ["var0", "var1"]:
    assert serial_multi_df[name].notna().sum() == parallel_multi_df[name].notna().sum()

print("Multi-variable serial and parallel results are identical in logical structure and occupancy.")

The parallel files remain in `tutorial4_output/`. This lets the same workflow scale beyond RAM while standard `xarray` and Dask readers consume the generated chunks.